In [3]:
# Bibliotecas
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment


In [4]:
# Ler os 3 Extratos dos bancos

arquivos = ["inter_agosto.csv", "nubank_agosto.csv", "santander_agosto.csv"]

df_inter = pd.read_csv('inter_agosto.csv')
df_nubank = pd.read_csv('nubank_agosto.csv')
df_santander = pd.read_csv('santander_agosto.csv')

# Criando o Tipo de Banco

df_inter["Banco"] = "Inter"
df_nubank["Banco"] = "Nubank"
df_santander["Banco"] = "Santander"

# Organizando a coluna de Data

df_inter = df_inter.rename(columns={'Data Lançamento': 'Data'})
df_nubank = df_nubank.rename(columns={'Data': 'Data'})
df_santander = df_santander.rename(columns={'Data Movimento': 'Data'})

# Organizando a coluna de Histórico

df_nubank = df_nubank.rename(columns={'Descrição': 'Histórico'})
df_santander = df_santander.rename(columns={'Descrição': 'Histórico'})

# Organizando a Coluna de Valores 

df_inter["Valor"] = (df_inter["Crédito"].fillna(df_inter["Débito"]))
df_inter = df_inter.drop(columns=["Crédito", "Débito"])



# Concatenando os três DataFrames


df_total = (pd.concat([df_inter, df_nubank, df_santander], ignore_index=True))

df_total = df_total.sort_values(by='Data')

# Adicionando Colunas por condição

df_total["Categoria"] = df_total['Histórico']

df_total["Tipo"] = df_total["Valor"].apply(lambda x: "Entrada" if x > 0 else "Saída")

# Criando a Coluna Categoria e Arrumando seus Dados
 
df_total["Categoria"] = "Outros"

 
df_total.loc[df_total["Histórico"].str.contains("SALÁRIO", case=False, na=False), "Categoria"] = "Salário"
df_total.loc[df_total["Histórico"].str.contains("NETFLIX|SPOTIFY", case=False, na=False), "Categoria"] = "Assinaturas"
df_total.loc[df_total["Histórico"].str.contains("UBER", case=False, na=False), "Categoria"] = "Transporte"
df_total.loc[df_total["Histórico"].str.contains("IFOOD", case=False, na=False), "Categoria"] = "Delivery"
df_total.loc[df_total["Histórico"].str.contains("SUPERMERCADO", case=False, na=False), "Categoria"] = "Alimentação"
df_total.loc[df_total["Histórico"].str.contains("TRANSFERÊNCIA|PIX", case=False, na=False), "Categoria"] = "Transferência"
df_total.loc[df_total["Histórico"].str.contains("POSTO", case=False, na=False), "Categoria"] = "Gasolina"
df_total.loc[df_total["Histórico"].str.contains("RENDIMENTO", case=False, na=False), "Categoria"] = "Investimento"

# Na Coluna Histórico iremos reduzir apenas para um valor para não triplicarmos o salário

df_total.drop(index=[0, 14], inplace=True)

df_total.loc[5, "Banco"] = "Origem"

# Apenas Visualização do DataFrame

display(df_total)


# Abrir os Extratos em 1 Planilha de excel

df_total.to_excel("excel_extrato.xlsx", index=False)
wb = load_workbook("excel_extrato.xlsx")
ws = wb.active

# Centralizar Todas as celulas 

for linha in ws.iter_rows():
    for celula in linha:
        celula.alignment = Alignment(horizontal="center",vertical="center")

wb.save("excel_extrato.xlsx")












,Data,Histórico,Banco,Valor,Tipo,Categoria
5,01/08/2026,SALÁRIO EMPRESA BETA ONLINE,Origem,5800.00,Entrada,Salário
6,01/08/2026,NETFLIX.COM,Nubank,-39.90,Saída,Assinaturas
1,02/08/2026,TRANSFERÊNCIA RECEBIDA,Inter,450.00,Entrada,Transferência
2,02/08/2026,SUPERMERCADO MUFFATO,Inter,-352.18,Saída,Alimentação
15,02/08/2026,PIX RECEBIDO CLIENTE,Santander,700.00,Entrada,Transferência
7,02/08/2026,IFOOD PEDIDO 983421,Nubank,-62.50,Saída,Delivery
8,02/08/2026,SPOTIFY,Nubank,-21.90,Saída,Assinaturas
16,03/08/2026,SUPERMERCADO CONFIANÇA,Santander,-312.90,Saída,Alimentação
9,03/08/2026,UBER TRIP,Nubank,-28.40,Saída,Transporte
10,03/08/2026,SUPERMERCADO CONFIANÇA,Nubank,-285.60,Saída,Alimentação
